# 자동매매 데이터 수집 1단계

목표: KOSPI, KOSDAQ 시가총액 상위 30개 종목을 가져오고, 다음 단계에서 학습용 가격 데이터를 수집할 수 있게 종목 리스트를 저장합니다.

In [ ]:
# 필요한 패키지 설치 확인
# 처음 실행할 때 FinanceDataReader가 없으면 아래 줄의 주석을 풀고 실행하세요.
# !pip install finance-datareader

from pathlib import Path
from datetime import datetime

import pandas as pd
import FinanceDataReader as fdr

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

TODAY = datetime.now().strftime("%Y%m%d")
TODAY

## 1. KOSPI, KOSDAQ 시총 상위 30개 가져오기

In [ ]:
def get_top_market_cap(market: str, n: int = 30) -> pd.DataFrame:
    """FinanceDataReader에서 시장별 시가총액 상위 n개 종목을 가져옵니다."""
    df = fdr.StockListing(market).copy()
    df = df.sort_values("Marcap", ascending=False).head(n).reset_index(drop=True)
    df.insert(0, "Rank", range(1, len(df) + 1))
    return df[[
        "Rank", "Code", "Name", "Market", "Close", "Changes", "ChagesRatio",
        "Volume", "Amount", "Marcap", "Stocks"
    ]]


kospi_top30 = get_top_market_cap("KOSPI", 30)
kosdaq_top30 = get_top_market_cap("KOSDAQ", 30)

display(kospi_top30)
display(kosdaq_top30)

## 2. 종목 리스트 저장하기

In [ ]:
top60 = pd.concat([kospi_top30, kosdaq_top30], ignore_index=True)

kospi_path = DATA_DIR / f"kospi_top30_marketcap_{TODAY}.csv"
kosdaq_path = DATA_DIR / f"kosdaq_top30_marketcap_{TODAY}.csv"
top60_path = DATA_DIR / f"krx_top60_marketcap_{TODAY}.csv"

kospi_top30.to_csv(kospi_path, index=False, encoding="utf-8-sig")
kosdaq_top30.to_csv(kosdaq_path, index=False, encoding="utf-8-sig")
top60.to_csv(top60_path, index=False, encoding="utf-8-sig")

print(kospi_path)
print(kosdaq_path)
print(top60_path)

## 3. 다음 단계용: 상위 종목 일봉 데이터 수집

모델 학습에는 시총 순위만으로는 부족하므로, 상위 60개 종목의 일봉 OHLCV 데이터도 함께 저장합니다.

In [ ]:
START_DATE = "2020-01-01"
END_DATE = datetime.now().strftime("%Y-%m-%d")

price_dir = DATA_DIR / "daily_prices"
price_dir.mkdir(exist_ok=True)

price_frames = []

for row in top60[["Code", "Name", "Market"]].itertuples(index=False):
    code, name, market = row
    try:
        price = fdr.DataReader(code, START_DATE, END_DATE).reset_index()
        price.insert(0, "Code", code)
        price.insert(1, "Name", name)
        price.insert(2, "Market", market)
        price_frames.append(price)

        file_path = price_dir / f"{market}_{code}_{name}.csv"
        price.to_csv(file_path, index=False, encoding="utf-8-sig")
    except Exception as exc:
        print(f"수집 실패: {market} {code} {name} - {exc}")

daily_prices = pd.concat(price_frames, ignore_index=True)
daily_prices_path = DATA_DIR / f"krx_top60_daily_prices_{TODAY}.csv"
daily_prices.to_csv(daily_prices_path, index=False, encoding="utf-8-sig")

print(daily_prices.shape)
print(daily_prices_path)
display(daily_prices.head())